In [1]:
# Import Libraries

import os
import random
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model

warnings.filterwarnings("ignore")

print("TensorFlow Version :", tf.__version__)

TensorFlow Version : 2.20.0


In [2]:
#Set Random Seed 

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
# Dataset Paths 

DATASET_PATH = "/kaggle/input/datasets/moltean/fruits/fruits-360_100x100/fruits-360"

TRAIN_PATH = os.path.join(DATASET_PATH, "Training")
TEST_PATH = os.path.join(DATASET_PATH, "Test")

print(TRAIN_PATH)
print(TEST_PATH)

/kaggle/input/datasets/moltean/fruits/fruits-360_100x100/fruits-360/Training
/kaggle/input/datasets/moltean/fruits/fruits-360_100x100/fruits-360/Test


In [4]:
# Classes to Keep

KEEP_CLASSES = [
    "Apple",
    "Banana",
    "Orange",
    "Mango",
    "Pear",
    "Kiwi",
    "Grape",
    "Strawberry",
    "Watermelon",
    "Pineapple",
    "Avocado",
    "Tomato",
    "Potato",
    "Onion",
    "Carrot",
    "Cucumber",
    "Pepper",
    "Cabbage",
    "Broccoli",
    "Corn",
    "Garlic",
    "Ginger",
    "Eggplant",
    "Lemon"
]

In [5]:
# Working Directory

WORK_DIR = "/kaggle/working"

TRAIN_NEW = os.path.join(WORK_DIR, "Training")

TEST_NEW = os.path.join(WORK_DIR, "Test")

os.makedirs(TRAIN_NEW, exist_ok=True)
os.makedirs(TEST_NEW, exist_ok=True)

In [6]:
# Copy Images into Merged Categories 

def copy_images(source_root, destination_root):

    for folder in sorted(os.listdir(source_root)):

        folder_path = os.path.join(source_root, folder)

        if not os.path.isdir(folder_path):
            continue
        matched = False

        for category in KEEP_CLASSES:

            if folder.startswith(category):

                target_folder = os.path.join(destination_root, category)

                os.makedirs(target_folder, exist_ok=True)

                for image in os.listdir(folder_path):

                    shutil.copy2(
                            os.path.join(folder_path, image),
                            os.path.join(target_folder, image)
                        )

                    matched = True
                    break

copy_images(TRAIN_PATH, TRAIN_NEW)

copy_images(TEST_PATH, TEST_NEW)

print("Finished copying dataset.")

Finished copying dataset.


In [7]:
# classes = sorted(os.listdir(TRAIN_NEW))

# print("Number of Categories :", len(classes))

# print(classes)

In [8]:
# image_count = {}

# for cls in classes:

#     folder = os.path.join(TRAIN_NEW, cls)

#     image_count[cls] = len(os.listdir(folder))

# df = pd.DataFrame(
#     image_count.items(),
#     columns=["Category","Images"]
# )

# df = df.sort_values("Images", ascending=False)

# df

In [9]:
# plt.figure(figsize=(14,7))

# plt.bar(df["Category"], df["Images"])

# plt.xticks(rotation=90)

# plt.title("Images per Category")

# plt.show()

In [10]:
# plt.figure(figsize=(18,12))

# for i, cls in enumerate(classes[:16]):

#     image_name = random.choice(
#         os.listdir(os.path.join(TRAIN_NEW, cls))
#     )

#     image_path = os.path.join(TRAIN_NEW, cls, image_name)

#     img = Image.open(image_path)

#     plt.subplot(4,4,i+1)

#     plt.imshow(img)

#     plt.title(cls)

#     plt.axis("off")

# plt.tight_layout()

# plt.show()

In [11]:
# Image Size 

IMAGE_SIZE = 224

BATCH_SIZE = 32

EPOCHS = 20

In [12]:
# Data Augmentation

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.20,
    width_shift_range=0.20,
    height_shift_range=0.20,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [13]:
# Data Generators 

train_generator = train_datagen.flow_from_directory(
    TRAIN_NEW,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=42
)

test_generator = test_datagen.flow_from_directory(
    TEST_NEW,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

Found 64 images belonging to 22 classes.
Found 82 images belonging to 22 classes.


In [14]:
# class_indices = train_generator.class_indices

# print(class_indices)

In [15]:
# images, labels = next(train_generator)

# print(images.shape)

# print(labels.shape)

In [16]:
#  Required Callbacks 

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

In [17]:
#  EfficientNetB0

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)

base_model.trainable = False

print("Base model loaded successfully.")

I0000 00:00:1784275118.359419      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784275118.362654      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Base model loaded successfully.


In [18]:
#  Build Classification Head

x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dropout(0.30)(x)

x = Dense(
    256,
    activation="relu"
)(x)

x = Dropout(0.30)(x)

outputs = Dense(
    train_generator.num_classes,
    activation="softmax"
)(x)

model = Model(
    inputs=base_model.input,
    outputs=outputs
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,383,161 (16.72 MB)

 Trainable params: 333,590 (1.27 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [19]:
# Compile Model 

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [20]:
#  Define Callbacks

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "/kaggle/working/best_food_category_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

In [21]:
# Train the Classification Head

history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=EPOCHS,
    callbacks=[
        early_stop,
        reduce_lr,
        checkpoint
    ]
)

Epoch 1/20


2026-07-17 07:59:04.055373: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-17 07:59:04.199605: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-17 07:59:04.544423: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-17 07:59:04.686151: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-17 07:59:04.827619: E external/local_xla/xla/stream_

1/2 ━━━━━━━━━━━━━━━━━━━━ 30s 31s/step - accuracy: 0.0938 - loss: 3.1468

2026-07-17 07:59:25.941137: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-17 07:59:26.082452: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-17 07:59:26.409695: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-17 07:59:26.550649: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-17 07:59:27.348108: E external/local_xla/xla/stream_


Epoch 1: val_accuracy improved from None to 0.14634, saving model to /kaggle/working/best_food_category_model.keras

Epoch 1: finished saving model to /kaggle/working/best_food_category_model.keras
2/2 ━━━━━━━━━━━━━━━━━━━━ 49s 19s/step - accuracy: 0.0781 - loss: 3.2198 - val_accuracy: 0.1463 - val_loss: 2.8748 - learning_rate: 0.0010
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 339ms/step - accuracy: 0.2031 - loss: 2.9262
Epoch 2: val_accuracy improved from 0.14634 to 0.26829, saving model to /kaggle/working/best_food_category_model.keras

Epoch 2: finished saving model to /kaggle/working/best_food_category_model.keras
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step - accuracy: 0.2188 - loss: 2.8976 - val_accuracy: 0.2683 - val_loss: 2.5894 - learning_rate: 0.0010
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step - accuracy: 0.2656 - loss: 2.6791
Epoch 3: val_accuracy improved from 0.26829 to 0.37805, saving model to /kaggle/working/best_food_category_model.keras

Epoch 3: finished saving model to /kag

In [22]:
# base_model.trainable = True

# fine_tune_at = len(base_model.layers) - 30

# for layer in base_model.layers[:fine_tune_at]:
#     layer.trainable = False

# print("Trainable Layers:",
#       len([l for l in model.layers if l.trainable]))

In [23]:
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(
#         learning_rate=1e-5
#     ),
#     loss="categorical_crossentropy",
#     metrics=["accuracy"]
# )

In [24]:
# fine_tune_history = model.fit(
#     train_generator,
#     validation_data=test_generator,
#     epochs=10,
#     callbacks=[
#         early_stop,
#         reduce_lr,
#         checkpoint
#     ]
# )

In [25]:
#  Save Final Model

model.save(
    "/kaggle/working/food_category_model.keras"
)

print("Model saved successfully.")

Model saved successfully.


In [26]:
# train_acc = history.history["accuracy"] + fine_tune_history.history["accuracy"]
# val_acc = history.history["val_accuracy"] + fine_tune_history.history["val_accuracy"]

# train_loss = history.history["loss"] + fine_tune_history.history["loss"]
# val_loss = history.history["val_loss"] + fine_tune_history.history["val_loss"]

# epochs = range(1, len(train_acc) + 1)

# plt.figure(figsize=(10,5))

# plt.plot(epochs, train_acc, label="Training Accuracy")
# plt.plot(epochs, val_acc, label="Validation Accuracy")

# plt.xlabel("Epoch")
# plt.ylabel("Accuracy")
# plt.title("Training vs Validation Accuracy")

# plt.legend()

# plt.show()

In [27]:
# plt.figure(figsize=(10,5))

# plt.plot(epochs, train_loss, label="Training Loss")
# plt.plot(epochs, val_loss, label="Validation Loss")

# plt.xlabel("Epoch")
# plt.ylabel("Loss")
# plt.title("Training vs Validation Loss")

# plt.legend()

# plt.show()

In [28]:
# import numpy as np
# import matplotlib.pyplot as plt

# from sklearn.metrics import (
#     classification_report,
#     confusion_matrix,
#     ConfusionMatrixDisplay
# )

In [29]:
# Evaluate Model

test_loss, test_accuracy = model.evaluate(
    test_generator,
    verbose=1
)

print("="*40)
print(f"Test Accuracy : {test_accuracy:.4f}")
print(f"Test Loss     : {test_loss:.4f}")
print("="*40)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.7195 - loss: 0.9980
Test Accuracy : 0.7195
Test Loss     : 0.9980


In [30]:
# test_generator.reset()

# predictions = model.predict(
#     test_generator,
#     verbose=1
# )

# predicted_classes = np.argmax(
#     predictions,
#     axis=1
# )

# true_classes = test_generator.classes

# class_names = list(test_generator.class_indices.keys())

In [31]:
# print(
#     classification_report(
#         true_classes,
#         predicted_classes,
#         target_names=class_names
#     )
# )

In [32]:
# cm = confusion_matrix(
#     true_classes,
#     predicted_classes
# )

# disp = ConfusionMatrixDisplay(
#     confusion_matrix=cm,
#     display_labels=class_names
# )

# fig, ax = plt.subplots(figsize=(18,18))

# disp.plot(
#     ax=ax,
#     xticks_rotation=90,
#     cmap="Blues",
#     colorbar=False
# )

# plt.title("Food Category Confusion Matrix")

# plt.show()

In [33]:
# import json

# class_mapping = {
#     value: key
#     for key, value in train_generator.class_indices.items()
# }

# with open(
#     "/kaggle/working/class_mapping.json",
#     "w"
# ) as f:
#     json.dump(
#         class_mapping,
#         f,
#         indent=4
#     )

# print(class_mapping)

In [34]:

#  Prediction Function 

from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet import preprocess_input

def predict_food(img_path):

    img = image.load_img(
        img_path,
        target_size=(224,224)
    )

    img_array = image.img_to_array(img)

    img_array = np.expand_dims(
        img_array,
        axis=0
    )

    img_array = preprocess_input(img_array)

    prediction = model.predict(
        img_array,
        verbose=0
    )

    predicted_index = np.argmax(prediction)

    predicted_class = class_mapping[predicted_index]

    confidence = np.max(prediction)

    return predicted_class, confidence

In [35]:
# sample_folder = random.choice(classes)

# sample_image = random.choice(
#     os.listdir(
#         os.path.join(
#             TEST_NEW,
#             sample_folder
#         )
#     )
# )

# sample_path = os.path.join(
#     TEST_NEW,
#     sample_folder,
#     sample_image
# )

# prediction, confidence = predict_food(
#     sample_path
# )

# img = Image.open(sample_path)

# plt.figure(figsize=(5,5))

# plt.imshow(img)

# plt.axis("off")

# plt.title(
#     f"Prediction : {prediction}\nConfidence : {confidence:.2%}"
# )

# plt.show()

# print("Actual Class :", sample_folder)
# print("Predicted    :", prediction)

In [36]:
# def top5_predictions(img_path):

#     img = image.load_img(
#         img_path,
#         target_size=(224,224)
#     )

#     img_array = image.img_to_array(img)

#     img_array = np.expand_dims(
#         img_array,
#         axis=0
#     )

#     img_array = preprocess_input(img_array)

#     prediction = model.predict(
#         img_array,
#         verbose=0
#     )[0]

#     top5 = prediction.argsort()[-5:][::-1]

#     print("Top 5 Predictions\n")

#     for idx in top5:

#         print(
#             f"{class_mapping[idx]:20s}"
#             f"{prediction[idx]*100:.2f}%"
#         )

In [37]:
# top5_predictions(sample_path)

In [38]:
# model.save(
#     "/kaggle/working/food_category_model.keras"
# )

# print("Model Saved Successfully")

In [39]:
# history_dict = {
#     "accuracy": train_acc,
#     "val_accuracy": val_acc,
#     "loss": train_loss,
#     "val_loss": val_loss
# }

# import pickle

# with open(
#     "/kaggle/working/training_history.pkl",
#     "wb"
# ) as f:

#     pickle.dump(
#         history_dict,
#         f
#     )

# print("Training history saved.")